In [31]:
import pandas as pd

In [32]:
# Load dataset
df = pd.read_csv("job_salary_prediction_dataset.csv")

# Preview data
df.head()

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069


In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   job_title         250000 non-null  str  
 1   experience_years  250000 non-null  int64
 2   education_level   250000 non-null  str  
 3   skills_count      250000 non-null  int64
 4   industry          250000 non-null  str  
 5   company_size      250000 non-null  str  
 6   location          250000 non-null  str  
 7   remote_work       250000 non-null  str  
 8   certifications    250000 non-null  int64
 9   salary            250000 non-null  int64
dtypes: int64(4), str(6)
memory usage: 19.1 MB


In [34]:
df.columns

Index(['job_title', 'experience_years', 'education_level', 'skills_count',
       'industry', 'company_size', 'location', 'remote_work', 'certifications',
       'salary'],
      dtype='str')

In [35]:
print(df.columns.tolist())

['job_title', 'experience_years', 'education_level', 'skills_count', 'industry', 'company_size', 'location', 'remote_work', 'certifications', 'salary']


In [36]:
# Normalisasi nama kolom
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

# Hapus missing values
df = df.dropna()

# Hapus duplikat
df = df.drop_duplicates()

# Validasi data salary (tidak boleh <= 0)
df = df[df['salary'] > 0]

# Cek ulang
df.shape

(250000, 10)

In [37]:
# Pilih kolom penting (sesuaikan jika nama berbeda)
columns_needed = [
    'job_title',
    'experience_years',
    'education_level',
    'industry',
    'company_size',
    'location',
    'salary'
]

df = df[columns_needed]

df.head()

,job_title,experience_years,education_level,industry,company_size,location,salary
0,AI Engineer,10,Bachelor,Healthcare,Medium,India,109413
1,Data Analyst,5,Bachelor,Telecom,Small,Australia,93764
2,Frontend Developer,18,PhD,Media,Medium,Singapore,148123
3,Business Analyst,19,PhD,Retail,Medium,Canada,189123
4,Product Manager,15,Bachelor,Manufacturing,Large,Sweden,165069


In [38]:
# Kategori gaji
def salary_category(salary):
    if salary < 50000:
        return "Low"
    elif salary < 100000:
        return "Medium"
    else:
        return "High"

df['salary_category'] = df['salary'].apply(salary_category)

df.head()

,job_title,experience_years,education_level,industry,company_size,location,salary,salary_category
0,AI Engineer,10,Bachelor,Healthcare,Medium,India,109413,High
1,Data Analyst,5,Bachelor,Telecom,Small,Australia,93764,Medium
2,Frontend Developer,18,PhD,Media,Medium,Singapore,148123,High
3,Business Analyst,19,PhD,Retail,Medium,Canada,189123,High
4,Product Manager,15,Bachelor,Manufacturing,Large,Sweden,165069,High


In [39]:
df.to_excel("cleaned_job_salary.xlsx", index=False)

print("✅ Data berhasil disimpan ke Excel")

✅ Data berhasil disimpan ke Excel


In [26]:
file_name = "job_salary.sql"

with open(file_name, "w") as f:
    # CREATE TABLE
    f.write("DROP TABLE IF EXISTS job_salary;\n\n")
    f.write("CREATE TABLE job_salary (\n")
    
    for col in df.columns:
        f.write(f"    {col} TEXT,\n")
    
    f.write(");\n\n")

    # INSERT DATA
    for _, row in df.iterrows():
        values = []
        for val in row:
            val = str(val).replace("'", "''")
            values.append(f"'{val}'")
        
        f.write(f"INSERT INTO job_salary VALUES ({', '.join(values)});\n")

print("✅ File SQL berhasil dibuat")

✅ File SQL berhasil dibuat


In [40]:
data_dict = pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str),
    "description": [
        "Judul pekerjaan",
        "Level pengalaman (tahun)",
        "Tingkat pendidikan",
        "Industri pekerjaan",
        "Ukuran perusahaan",
        "Lokasi kerja",
        "Gaji",
        "Kategori gaji"
    ]
})

data_dict.to_excel("data_dictionary.xlsx", index=False)

data_dict

,column,data_type,description
job_title,job_title,str,Judul pekerjaan
experience_years,experience_years,int64,Level pengalaman (tahun)
education_level,education_level,str,Tingkat pendidikan
industry,industry,str,Industri pekerjaan
company_size,company_size,str,Ukuran perusahaan
location,location,str,Lokasi kerja
salary,salary,int64,Gaji
salary_category,salary_category,str,Kategori gaji


In [41]:
# Rata-rata gaji berdasarkan pengalaman
df.groupby('experience_years')['salary'].mean().sort_values(ascending=False)

experience_years
20    173179.709650
19    169797.362442
18    167421.185837
17    164282.181423
16    162290.866980
15    158981.141636
14    156413.152885
13    154082.616121
12    150779.033238
11    148133.173236
10    145771.972998
9     142763.834683
8     140667.762731
7     137560.810366
6     134898.296712
5     131787.250717
4     129688.783435
3     126921.603112
2     124228.943849
1     121250.527183
0     118872.622755
Name: salary, dtype: float64

In [42]:
# Top job dengan gaji tertinggi
df.groupby('job_title')['salary'].mean().sort_values(ascending=False).head(10)

job_title
AI Engineer                  173498.480640
Machine Learning Engineer    163022.504570
Product Manager              157594.932029
Cloud Engineer               152102.535290
DevOps Engineer              149959.266791
Cybersecurity Analyst        148697.695548
Data Scientist               147258.214409
Software Engineer            141739.521460
Backend Developer            139202.768663
Frontend Developer           132653.842485
Name: salary, dtype: float64

In [43]:
# Rata-rata gaji per industri
df.groupby('industry')['salary'].mean().sort_values(ascending=False).head(10)

industry
Education        145993.564547
Media            145891.271071
Telecom          145876.511967
Technology       145863.808377
Finance          145801.639468
Healthcare       145759.995702
Government       145613.869242
Manufacturing    145530.603301
Consulting       145451.638293
Retail           145399.699408
Name: salary, dtype: float64